In [4]:
import os
import shutil
import glob
import numpy as np
import tqdm
import time
import re

In [5]:
# reading the file names from the experiment folders
experiment_no = 423
no_channels = 10 # exclude the clock channel
indices_to_exclude = [] # excluding these indices from the experiment to remove the parts where we put in and take out the animal

In [6]:
# Functions to sort file names correctly
def atoi(text):
    return int(text) if text.isdigit() else text

def natural_keys(text):
    return [atoi(c) for c in re.split(r'(\d+)', text)]

### Making the average channels for DAS

In [7]:
from scipy.io import wavfile
import pandas as pd

In [8]:
general_path = "D:/big_setup/experiment_{}/concatenated_data_cam_mic_sync/".format(experiment_no)
# Getting the video files - assumes one camera
vid_files = glob.glob(general_path+"*.mp4")
vid_files.sort(key=natural_keys)

# Getting the channels
channel_files = {}
for idx in range(no_channels):
    temp_str = "%02d"%(idx)
    channel_files[idx] = glob.glob(general_path+f"channel_{temp_str}_*.wav")
    channel_files[idx].sort(key=natural_keys)

In [9]:
files  = pd.DataFrame.from_dict(channel_files)
for index, row in tqdm.tqdm(files.iterrows()):
    data = []
    for wav in row:
        sr, temp = wavfile.read(wav)
        data.append(temp)
    idx_str = "%03d"%(index)
    file_name = f"Z:/Niegil/das/mono_channels/exp_{experiment_no}_idx_{idx_str}_ch_all.wav"
    wavfile.write(file_name, sr, np.array(data).mean(axis=0).astype("float32"))


15it [00:41,  2.76s/it]


### After DAS and SLEAP is done

In [47]:


general_path = "D:/big_setup/experiment_{}/concatenated_data_cam_mic_sync/".format(experiment_no)
# Getting the video files - assumes one camera
vid_files = glob.glob(general_path+"*.mp4")
vid_files.sort(key=natural_keys)

# Getting the channels
channel_files = {}
for idx in range(no_channels):
    temp_str = "%02d"%(idx)
    channel_files[idx] = glob.glob(general_path+f"channel_{temp_str}_*.wav")
    channel_files[idx].sort(key=natural_keys)

# getting the tracks
track_files = glob.glob(general_path+"tracks/*.csv")
track_files.sort(key=natural_keys)

# getting the das files
das_files = glob.glob(f"//sanesstorage.cns.nyu.edu/archive/Niegil/das/training_data/data/exp_{experiment_no}*annotations.csv")
das_files.sort(key=natural_keys)

In [48]:
if (len(das_files) <= 0): 
    raise Exception("Das files not found")

if (len(track_files) <= 0): 
    raise Exception("Track annotations files not found")

if (len(channel_files) <= 0): 
    raise Exception("Channel files not found")

if (len(vid_files) <= 0): 
    raise Exception("Video files not found")

In [49]:
length_of_experiment = len(glob.glob(general_path+"*.wav"))//(no_channels+1)

In [50]:
# creating the folders
for i in range(length_of_experiment-len(indices_to_exclude)):
    try:
        temp_str = "%03d"%(i)
        os.makedirs(general_path+f"ssl_data_path/idx_{temp_str}")
    except Exception as e:
        # print(e)
        pass

In [ ]:
ssl_data_idx = 0
folders_to_delete = []
for idx in tqdm.tqdm(range(length_of_experiment)):
    # if idx in indices_to_exclude:
    #     continue
    temp_str = "%03d"%(ssl_data_idx)
    
    # copying channel files
    for i in range(no_channels):
        file = channel_files[i][idx]
        file_name = file.split("\\")[-1]
        if "%03d"%(idx) in file:
            shutil.copy(file,general_path+f"ssl_data_path/idx_{temp_str}/{file_name}")
            # pass
        else:
            print(f"Error copying {file} to idx {idx}")
    
    # copying video files
    index = [id for id, s in enumerate(vid_files) if "%03d"%(idx) in s]
    if len(index) == 1:
        file = vid_files[index[0]]
        file_name = file.split("\\")[-1]
        shutil.copy(file,general_path+f"ssl_data_path/idx_{temp_str}/{file_name}")
        # pass
    else:
        print(f"Error copying {file} to idx {idx}")
        folders_to_delete.append(general_path+f"ssl_data_path/idx_{temp_str}")
        ssl_data_idx+=1
        continue

    # copying das files
    index = [id for id, s in enumerate(das_files) if "%03d"%(idx) in s]
    if len(index) == 1:
        file = das_files[index[0]]
        file_name = file.split("\\")[-1]
        shutil.copy(file,general_path+f"ssl_data_path/idx_{temp_str}/{file_name}")
        # pass
    else:
        print(f"Number of DAS files found for index {idx} is {len(index)}")
        print(f"More than one or less than one of das annotations files found for index {idx}")
        folders_to_delete.append(general_path+f"ssl_data_path/idx_{temp_str}/{file_name}")
        ssl_data_idx+=1
        continue
    
    # copying track files
    index = [id for id, s in enumerate(track_files) if "center_"+"%03d"%(idx) in s]
    if len(index) == 1:
        file = track_files[index[0]]
        file_name = file.split("\\")[-1]
        # file_name = file_name.split('.')[2]+".tracks.csv"
        shutil.copy(file,general_path+f"ssl_data_path/idx_{temp_str}")
        # print(file,file_name)
        # pass
    else:
        print(f"Number of track files found for index {idx} is {len(index)}")
        print(f"More than one or less than one of track files found for index {idx}")
        folders_to_delete.append(general_path+f"ssl_data_path/idx_{temp_str}")
        ssl_data_idx+=1
        continue


    ssl_data_idx+=1


 87%|████████▋ | 78/90 [14:58<01:15,  6.32s/it]

In [ ]:
# deleting the folders with issues
for folder in folders_to_delete:
        if os.path.exists(folder):
            shutil.rmtree(folder)
            print(f"Directory {folder} and its contents deleted successfully.")

: 

: 

: 